<a href="https://colab.research.google.com/github/AfraazUlHaque/ASSIGNMENTS/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfraazUlHaque/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My Lane as an ML Task

My chosen lane is **Refresh / Content Opportunity Scoring**.

The primary ML task is **ranking / scoring**. The goal is to assign each eligible content page a priority score so that a content editor or SEO analyst can review the most promising or risky pages first.

A classification model may be used as one component of the ranking, for example by estimating the probability that a page belongs to a decline-risk group. However, the final product is not only a yes/no prediction — it is an ordered review queue.

The action supported by this output is deciding which pages should be reviewed first for possible refresh, expansion, CTR review, protection, or monitoring.

In [ ]:
lane = "Refresh / Content Opportunity Scoring"
task_type = "Ranking / Scoring"

print("Lane:", lane)
print("Primary ML task:", task_type)

Lane: Refresh / Content Opportunity Scoring
Primary ML task: Ranking / Scoring


## 2. Target or Proxy

For the starter dataset, I will use a **proxy target** based on whether `trend_direction` is `"down"`.

This can be represented as:

`decline_proxy = 1` when `trend_direction == "down"`, otherwise `0`.

This is useful for learning the ML workflow, but it is not an ideal final target because `trend_direction` describes the current trailing window rather than a future observed outcome.

For a stronger capstone, I would prefer a future-looking target such as:

**features from a previous window → observed decline over a later window**

For example, past 90-day signals could be used to predict whether visibility declines in the following 30 days.

Because the starter proxy is derived from `trend_direction`, neither `trend_direction` nor `trend_pct` should be used as model features when predicting this proxy.

In [ ]:
import pandas as pd

# Load the CSV uploaded directly to Colab
df = pd.read_csv("content_refresh_anonymized.csv")

# Create proxy target
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)

# Show target distribution
print("Decline Proxy Distribution:")
print(df["decline_proxy"].value_counts())

# Show positive rate
print(
    "\nProxy positive rate:",
    round(df["decline_proxy"].mean() * 100, 2),
    "%"
)


Decline Proxy Distribution:
decline_proxy
1    16262
0    13738
Name: count, dtype: int64

Proxy positive rate: 54.21 %


## 3. Success Metric

My main success metric will be **Precision@K** because the final output is a ranked review queue and editorial capacity is limited.

For example, if an editor can review only 20 or 50 pages, Precision@20 or Precision@50 measures how many of those highest-ranked pages match the chosen target or proxy.

This metric is more useful here than generic accuracy because the practical decision is not to classify every page correctly. The practical decision is to make the top of the review queue as useful as possible.

I will also inspect Average Precision and recall where appropriate, but Precision@K is the primary decision-oriented metric.

In [ ]:
review_capacity = [20, 50]

for k in review_capacity:
    print(f"Primary evaluation candidate: Precision@{k}")

Primary evaluation candidate: Precision@20
Primary evaluation candidate: Precision@50


## 4. Unit of Analysis

The unit of analysis is **one pseudonymized content item (one page)**.

The starter dataset already has approximately one row per content item. For this lane, I keep pages with observable search demand and sufficient content age, following the starter workflow:

- `impressions_90d > 0`
- `content_age_days >= 90`

I also deduplicate by `content_id`.

Each resulting row represents one page that could potentially receive a review-priority score.

In [ ]:
lane_df = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

display_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "decline_proxy"
]

display(lane_df[display_columns].head(10))

print("Unit of analysis: one content item / page")
print("Eligible rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,ctr,avg_position,decline_proxy
0,content_304f48230142,client_f369cb89fc,3803,17,187,0.76,10.6,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,0.05,20.3,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,0.09,36.5,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,0.49,6.2,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,0.13,44.0,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,0.03,8.5,1
6,content_9a34b442b552,client_8722616204,20,1,90,0.00,7.0,1
7,content_a63219c6e95a,client_19581e27de,1724,28,445,0.06,21.2,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,0.09,46.0,1
9,content_c27558df2b0c,client_19581e27de,1240,3,257,0.16,4.9,1


Unit of analysis: one content item / page
Eligible rows: 30000
Unique content IDs: 30000


## 5. Why ML May Beat a Fixed Rule Here

A fixed rule is a necessary baseline. For example, a simple system could prioritize pages that are old, have high impressions, and show weak performance.

However, page priority may depend on several signals interacting at the same time, including visibility, CTR, position, sessions, content age, freshness, engagement, and content characteristics. A small set of hand-written thresholds may miss combinations of signals that are useful for ranking.

ML may help by learning these interactions and producing a smoother priority estimate instead of relying only on hard cutoffs.

However, ML has to earn its place. I will first compare it against a transparent rule-based baseline. If the model does not improve Precision@K or produce a more useful review queue under honest validation, the simpler rule should be preferred.

The model output remains decision-support. A high score means "review this page earlier," not "editing this page will definitely improve performance."

In [ ]:
print("Baseline first -> ML second -> compare using Precision@K")
print("Final action: rank pages for human editorial review")

Baseline first -> ML second -> compare using Precision@K
Final action: rank pages for human editorial review


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.